# Объяснятель: выравнивание Qwen3-4B-Instruct-2507

Полностью воспроизводимый Colab-пайплайн: SFT → style-DPO, отдельная Reward Model, затем независимые quality-DPO и SimPO.

## 0. Цель и инварианты

- Все random seeds равны `42`.
- Обучение выполняется через QLoRA 4-bit на T4.
- `public_test_*` не участвуют в обучении.
- Style-генерация выполняется с `do_sample=False`.
- Все пять метрик вычисляются внутри этого ноутбука.
- DPO и SimPO по качеству ветвятся от одного style-DPO чекпоинта.

## 1. Настройка окружения

In [ ]:
%pip install -q \
  transformers==4.57.3 trl==0.24.0 peft==0.17.1 \
  accelerate==1.11.0 bitsandbytes==0.48.2 datasets==4.3.0 \
  scikit-learn==1.7.2 safetensors==0.6.2 jinja2==3.1.6 \
  modelscope==1.36.1

In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

# Kaggle exposes two T4s; the 4-bit QLoRA pipeline intentionally uses one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import bitsandbytes as bnb
import datasets
import numpy as np
import peft
import sklearn
import torch
import transformers
import trl
from transformers import set_seed

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

assert torch.cuda.is_available(), "Для полного прогона требуется GPU T4"
print("Python:", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)
subprocess.run(["nvidia-smi"], check=True)

## 2. Данные и проверки

Загрузка официального архива, проверка SHA-256 и схем JSONL.

In [ ]:
ASSET_URL = "https://edu.tbank.ru/files/c1005bf0-8695-451a-9616-87c8687dce27"
RUNTIME_ROOT = Path("/content/alignment_explainer")
ARCHIVE_PATH = RUNTIME_ROOT / "ml-olympiad-red-task.zip"
EXTRACT_DIR = RUNTIME_ROOT / "source"
CHECKPOINT_DIR = RUNTIME_ROOT / "checkpoints"

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    print("Downloading organizer assets...")
    urllib.request.urlretrieve(ASSET_URL, ARCHIVE_PATH)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(EXTRACT_DIR)

kid_matches = list(EXTRACT_DIR.rglob("kid_adult.jsonl"))
assert len(kid_matches) == 1, f"Expected one kid_adult.jsonl, found {kid_matches}"
PROJECT_ROOT = kid_matches[0].parent.parent
DATA_DIR = PROJECT_ROOT / "data"
METRICS_DIR = PROJECT_ROOT / "metrics"
print("Assets root:", PROJECT_ROOT)

In [ ]:
EXPECTED_SHA256 = {
    "data/good_bad.jsonl": "bf50f3af0127df71d891c5a65eb75220104157f3e27b613aacbae1761c08998b",
    "data/kid_adult.jsonl": "52bacff1c6d5d50ca3dd473f8d754cf1dfcce7e02ecf162cda2c18719a138748",
    "data/public_test_quality.jsonl": "bc8b21bf04c88e99d420569c61f46309f71d04601159b80ea258760e8d871780",
    "data/public_test_style.jsonl": "d0f5fb848245b18e97b97fe5158c602f3f2b49b8ec6588f93a0f0e9f10c58efe",
    "metrics/style_clf.pkl": "b5cf7b53417033de19b9c44a43402bce0e6eeece44b1abac2cf596785b60888d",
    "metrics/gold_rm/adapter_config.json": "e323f8652b0d1b571163c21db0175ac32bee06bc48022b82cd1f7d7c1e94b8fd",
    "metrics/gold_rm/adapter_model.safetensors": "fbfa95e616bc88f6f17da81067390053c938e1e36448cf65b41499603ce2d704",
    "metrics/gold_rm/value_head.safetensors": "4feddd918c31985e644c143c21a49b6739731f6e2eb78e858801109196505f08",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for relative_path, expected_hash in EXPECTED_SHA256.items():
    asset_path = PROJECT_ROOT / relative_path
    assert asset_path.exists(), f"Missing asset: {relative_path}"
    actual_hash = sha256_file(asset_path)
    assert actual_hash == expected_hash, (relative_path, actual_hash, expected_hash)
print(f"Verified {len(EXPECTED_SHA256)} organizer assets by SHA-256")

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

kid_adult_rows = read_jsonl(DATA_DIR / "kid_adult.jsonl")
good_bad_rows = read_jsonl(DATA_DIR / "good_bad.jsonl")
public_style_rows = read_jsonl(DATA_DIR / "public_test_style.jsonl")
public_quality_rows = read_jsonl(DATA_DIR / "public_test_quality.jsonl")

EXPECTED_SCHEMAS = {
    "kid_adult": (kid_adult_rows, 1489, {"prompt", "kid", "adult"}),
    "good_bad": (good_bad_rows, 2226, {"instruction", "chosen", "rejected"}),
    "public_style": (public_style_rows, 50, {"prompt", "kid", "adult", "fact"}),
    "public_quality": (public_quality_rows, 50, {"prompt", "chosen", "rejected"}),
}
for name, (rows, expected_count, expected_keys) in EXPECTED_SCHEMAS.items():
    assert len(rows) == expected_count, (name, len(rows), expected_count)
    assert all(set(row) == expected_keys for row in rows), f"Unexpected schema in {name}"
    assert all(isinstance(value, str) and value.strip() for row in rows for value in row.values())

assert not ({row["prompt"] for row in kid_adult_rows} & {row["prompt"] for row in public_style_rows})
quality_group_sizes = Counter(row["instruction"] for row in good_bad_rows)
print({name: len(rows) for name, (rows, _, _) in EXPECTED_SCHEMAS.items()})
print("Unique quality instructions:", len(quality_group_sizes))
print("Pairs per quality instruction:", Counter(quality_group_sizes.values()))
print("Public datasets are evaluation-only and are not included in any training variable.")

## 3. Формальные измерители

Style probability, pairwise reward accuracy и length-normalized implicit-preference accuracy.

In [ ]:
import pickle
from scipy.sparse import hstack

with (METRICS_DIR / "style_clf.pkl").open("rb") as file:
    style_bundle = pickle.load(file)

style_classifier = style_bundle["clf"]
style_vectorizers = style_bundle["vecs"]
assert list(style_classifier.classes_) == [0, 1]
assert len(style_vectorizers) == 2

def simple_probabilities(texts: list[str]) -> np.ndarray:
    """Return P(simple) for response text only; prompts must not be included."""
    features = hstack(
        [vectorizer.transform(texts) for vectorizer in style_vectorizers],
        format="csr",
    )
    simple_index = list(style_classifier.classes_).index(1)
    return style_classifier.predict_proba(features)[:, simple_index]

sanity_rows = kid_adult_rows[:256]
kid_sanity = simple_probabilities([row["kid"] for row in sanity_rows]).mean()
adult_sanity = simple_probabilities([row["adult"] for row in sanity_rows]).mean()
assert kid_sanity > adult_sanity + 0.50, (kid_sanity, adult_sanity)
print(f"Style scorer sanity: kid={kid_sanity:.4f}, adult={adult_sanity:.4f}")

In [ ]:
def interval_letter(value: float, metric_kind: str) -> str:
    """Map metrics to the task's half-open intervals."""
    assert np.isfinite(value) and 0.0 <= value <= 1.0, value
    if metric_kind == "style":
        bounds = ((0.4, "А"), (0.7, "Б"), (0.9, "В"), (float("inf"), "Г"))
    elif metric_kind == "quality":
        bounds = ((0.6, "А"), (0.75, "Б"), (0.9, "В"), (float("inf"), "Г"))
    else:
        raise ValueError(f"Unknown metric kind: {metric_kind}")
    return next(letter for upper_bound, letter in bounds if value < upper_bound)

def strict_pairwise_accuracy(chosen_scores, rejected_scores) -> tuple[float, int, int]:
    chosen = np.asarray(chosen_scores, dtype=np.float64)
    rejected = np.asarray(rejected_scores, dtype=np.float64)
    assert chosen.shape == rejected.shape and chosen.ndim == 1 and chosen.size > 0
    correct = chosen > rejected  # ties are errors
    return float(correct.mean()), int(correct.sum()), int(correct.size)

METRICS = {}

def record_metric(task: str, value: float, metric_kind: str, **details) -> dict:
    result = {"value": float(value), "interval": interval_letter(value, metric_kind), **details}
    METRICS[task] = result
    print(f"{task}: {value:.6f} -> {result['interval']}")
    return result

assert [interval_letter(x, "quality") for x in (0.59, 0.60, 0.75, 0.90)] == ["А", "Б", "В", "Г"]

In [ ]:
import torch.nn.functional as F

def encode_prompt_completion(tokenizer, prompt: str, completion: str) -> tuple[list[int], list[bool]]:
    """Tokenize an exact chat and mark assistant content, excluding prompt and <|im_end|>."""
    prompt_messages = [{"role": "user", "content": prompt}]
    full_messages = prompt_messages + [{"role": "assistant", "content": completion}]
    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True
    )
    full_ids = tokenizer.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False
    )
    prompt_ids = list(prompt_ids)
    full_ids = list(full_ids)
    assert full_ids[: len(prompt_ids)] == prompt_ids, "Chat prompt is not a prefix of full conversation"

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    assert im_end_id != tokenizer.unk_token_id
    answer_start = len(prompt_ids)
    answer_end = full_ids.index(im_end_id, answer_start)
    assert answer_end > answer_start, "Empty assistant completion"
    completion_mask = [False] * len(full_ids)
    completion_mask[answer_start:answer_end] = [True] * (answer_end - answer_start)
    return full_ids, completion_mask

def mean_completion_logprobs(
    model, tokenizer, prompts: list[str], completions: list[str], *, batch_size: int = 2, max_length: int = 512
) -> np.ndarray:
    """Mean conditional logprob over assistant-content tokens only."""
    assert len(prompts) == len(completions) and prompts
    encoded = [encode_prompt_completion(tokenizer, p, c) for p, c in zip(prompts, completions)]
    too_long = [len(ids) for ids, _ in encoded if len(ids) > max_length]
    assert not too_long, f"Increase max_length; observed lengths: {too_long}"

    pad_token_id = tokenizer.pad_token_id
    assert pad_token_id is not None
    device = next(model.parameters()).device
    scores = []
    was_training = model.training
    model.eval()
    try:
        for start in range(0, len(encoded), batch_size):
            batch = encoded[start : start + batch_size]
            width = max(len(ids) for ids, _ in batch)
            input_ids = []
            attention_masks = []
            completion_masks = []
            for ids, mask in batch:
                padding = width - len(ids)
                input_ids.append(ids + [pad_token_id] * padding)
                attention_masks.append([1] * len(ids) + [0] * padding)
                completion_masks.append(mask + [False] * padding)

            input_tensor = torch.tensor(input_ids, dtype=torch.long, device=device)
            attention_tensor = torch.tensor(attention_masks, dtype=torch.long, device=device)
            answer_mask = torch.tensor(completion_masks, dtype=torch.bool, device=device)[:, 1:]
            with torch.inference_mode():
                logits = model(
                    input_ids=input_tensor, attention_mask=attention_tensor, use_cache=False
                ).logits[:, :-1, :]
                labels = input_tensor[:, 1:]
                token_nll = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)), labels.reshape(-1), reduction="none"
                ).view_as(labels)
                token_logprobs = -token_nll
                token_counts = answer_mask.sum(dim=1)
                assert torch.all(token_counts > 0)
                batch_scores = (token_logprobs * answer_mask).sum(dim=1) / token_counts
                scores.extend(batch_scores.float().cpu().tolist())
            del input_tensor, attention_tensor, answer_mask, logits, labels, token_nll
    finally:
        if was_training:
            model.train()
    return np.asarray(scores, dtype=np.float64)

## 4. Задача 1 — SFT

In [ ]:
from datasets import Dataset
from modelscope import snapshot_download
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
MODELSCOPE_REVISION = "master"
MODEL_SNAPSHOT_DIR = RUNTIME_ROOT / "qwen3_4b_instruct_2507"
MAX_LENGTH = 512
MAX_NEW_TOKENS = 192
SFT_ADAPTER_DIR = CHECKPOINT_DIR / "task1_sft"
STYLE_DPO_ADAPTER_DIR = CHECKPOINT_DIR / "task2_style_dpo"
RM_ADAPTER_DIR = CHECKPOINT_DIR / "task3_reward_model"
QUALITY_DPO_ADAPTER_DIR = CHECKPOINT_DIR / "task4_quality_dpo"
SIMPO_ADAPTER_DIR = CHECKPOINT_DIR / "task5_simpo"

QLORA_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

EXPECTED_MODEL_SHA256 = {
    "config.json": "5beea1a4a34c62782bfb2f911c606741a3bab8f92d80a118fa053c28af12e8ba",
    "generation_config.json": "835fffe355c9438e7a25be099b3fccaa98350b83451f9fd2d99512e74f1ade48",
    "model.safetensors.index.json": "d6c42883a895dfef5b0080ed2116a1bcd764f558406b98923d675978a1abf29c",
    "model-00001-of-00003.safetensors": "75311d91bb08cf0b882913da464a1e722a31fb44db35208663487efb7a3d8ed6",
    "model-00002-of-00003.safetensors": "0b48adbb1f60e901153d91907ba11ce63bd4b8b584482e730f48808d055dfba1",
    "model-00003-of-00003.safetensors": "7dd39ccca5e4de123c74c14af44c9bf2eb75df33b4614382af0134528e060d5d",
}

def official_hf_weights_available() -> bool:
    probe_url = (
        f"https://huggingface.co/{MODEL_ID}/resolve/{MODEL_REVISION}/"
        "model-00001-of-00003.safetensors"
    )
    request = urllib.request.Request(probe_url, headers={"Range": "bytes=0-0"})
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            return response.status in (200, 206) and bool(response.read(1))
    except Exception as error:
        print(f"Hugging Face weight endpoint unavailable ({type(error).__name__}); using verified ModelScope mirror.")
        return False

def prepare_model_source() -> str:
    if official_hf_weights_available():
        print("Model source: Hugging Face")
        return MODEL_ID
    snapshot_path = Path(snapshot_download(
        MODEL_ID,
        revision=MODELSCOPE_REVISION,
        local_dir=str(MODEL_SNAPSHOT_DIR),
        allow_patterns=list(EXPECTED_MODEL_SHA256),
        max_workers=2,
    ))
    for filename, expected_hash in EXPECTED_MODEL_SHA256.items():
        model_file = snapshot_path / filename
        assert model_file.exists(), model_file
        assert sha256_file(model_file) == expected_hash, filename
    print("Model source: verified ModelScope snapshot", snapshot_path)
    return str(snapshot_path)

MODEL_SOURCE = prepare_model_source()

def model_revision_kwargs() -> dict:
    return {"revision": MODEL_REVISION} if MODEL_SOURCE == MODEL_ID else {}

def load_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    assert tokenizer.chat_template
    return tokenizer

def load_qlora_causal_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_SOURCE,
        **model_revision_kwargs(),
        quantization_config=QLORA_CONFIG,
        device_map={"": 0},
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    return model

def causal_lora_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=LORA_TARGET_MODULES,
    )

In [ ]:
tokenizer = load_tokenizer()
sft_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "completion": [{"role": "assistant", "content": row["kid"]}],
    }
    for row in kid_adult_rows
])

sft_lengths = [
    len(tokenizer.apply_chat_template(record["prompt"] + record["completion"], tokenize=True))
    for record in sft_dataset
]
assert max(sft_lengths) <= MAX_LENGTH, (max(sft_lengths), MAX_LENGTH)
print("SFT examples:", len(sft_dataset))
print("SFT token lengths (min/median/max):", min(sft_lengths), int(np.median(sft_lengths)), max(sft_lengths))

In [ ]:
set_seed(SEED)
sft_base_model = load_qlora_causal_model()
sft_args = SFTConfig(
    output_dir=str(RUNTIME_ROOT / "trainer_task1_sft"),
    seed=SEED,
    data_seed=SEED,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    num_train_epochs=1.0,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataset_num_proc=2,
    eos_token="<|im_end|>",
)
sft_trainer = SFTTrainer(
    model=sft_base_model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=causal_lora_config(),
)
sft_trainer.model.print_trainable_parameters()
sft_train_result = sft_trainer.train()
sft_trainer.save_model(str(SFT_ADAPTER_DIR))
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print("SFT train loss:", sft_train_result.training_loss)

In [ ]:
def generate_greedy_responses(
    model, tokenizer, prompts: list[str], *, batch_size: int = 4
) -> list[str]:
    assert prompts and batch_size > 0
    model.eval()
    responses = []
    eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    try:
        for start in range(0, len(prompts), batch_size):
            batch_prompts = prompts[start:start + batch_size]
            rendered = [
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for prompt in batch_prompts
            ]
            model_inputs = tokenizer(
                rendered, padding=True, return_tensors="pt", add_special_tokens=False
            ).to(next(model.parameters()).device)
            input_length = model_inputs["input_ids"].shape[1]
            with torch.inference_mode():
                generated = model.generate(
                    **model_inputs,
                    do_sample=False,
                    num_beams=1,
                    max_new_tokens=MAX_NEW_TOKENS,
                    eos_token_id=eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    use_cache=True,
                )
            decoded = tokenizer.batch_decode(
                generated[:, input_length:], skip_special_tokens=True
            )
            for prompt, response in zip(batch_prompts, decoded):
                response = response.strip()
                assert response, f"Empty response for prompt: {prompt!r}"
                responses.append(response)
    finally:
        tokenizer.padding_side = original_padding_side
    return responses

task1_prompts = [row["prompt"] for row in public_style_rows]
task1_responses = generate_greedy_responses(sft_trainer.model, tokenizer, task1_prompts)
task1_probabilities = simple_probabilities(task1_responses)
task1_style_score = float(task1_probabilities.mean())
record_metric("task_1_sft_style", task1_style_score, "style", count=len(task1_responses))

with (RUNTIME_ROOT / "task1_responses.json").open("w", encoding="utf-8") as file:
    json.dump(
        [{"prompt": p, "response": r, "p_simple": float(s)} for p, r, s in zip(task1_prompts, task1_responses, task1_probabilities)],
        file, ensure_ascii=False, indent=2,
    )

del sft_trainer, sft_base_model
gc.collect()
torch.cuda.empty_cache()

## 5. Задача 2 — DPO по стилю

In [ ]:
from trl import DPOConfig, DPOTrainer

style_dpo_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "chosen": [{"role": "assistant", "content": row["kid"]}],
        "rejected": [{"role": "assistant", "content": row["adult"]}],
    }
    for row in kid_adult_rows
])
print("Style-DPO preference pairs:", len(style_dpo_dataset))

In [ ]:
set_seed(SEED)
style_dpo_base = load_qlora_causal_model()
style_dpo_model = PeftModel.from_pretrained(
    style_dpo_base, SFT_ADAPTER_DIR, is_trainable=True, adapter_name="default"
)
style_dpo_model.load_adapter(
    SFT_ADAPTER_DIR, adapter_name="reference", is_trainable=False
)
style_dpo_model.set_adapter("default")
assert set(style_dpo_model.peft_config) == {"default", "reference"}
style_dpo_model.print_trainable_parameters()

style_dpo_args = DPOConfig(
    output_dir=str(RUNTIME_ROOT / "trainer_task2_style_dpo"),
    seed=SEED,
    data_seed=SEED,
    model_adapter_name="default",
    ref_adapter_name="reference",
    beta=0.10,
    loss_type="sigmoid",
    max_length=MAX_LENGTH,
    num_train_epochs=1.0,
    max_steps=120,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataset_num_proc=2,
)
style_dpo_trainer = DPOTrainer(
    model=style_dpo_model,
    ref_model=None,
    args=style_dpo_args,
    train_dataset=style_dpo_dataset,
    processing_class=tokenizer,
)
style_dpo_result = style_dpo_trainer.train()
style_dpo_trainer.model.set_adapter("default")
style_dpo_trainer.model.save_pretrained(
    STYLE_DPO_ADAPTER_DIR, selected_adapters=["default"]
)
tokenizer.save_pretrained(STYLE_DPO_ADAPTER_DIR)
print("Style-DPO train loss:", style_dpo_result.training_loss)

In [ ]:
task2_responses = generate_greedy_responses(
    style_dpo_trainer.model, tokenizer, task1_prompts
)
task2_probabilities = simple_probabilities(task2_responses)
task2_style_score = float(task2_probabilities.mean())
record_metric("task_2_dpo_style", task2_style_score, "style", count=len(task2_responses))
print(f"Style-DPO delta vs SFT: {task2_style_score - task1_style_score:+.6f}")

with (RUNTIME_ROOT / "task2_responses.json").open("w", encoding="utf-8") as file:
    json.dump(
        [{"prompt": p, "response": r, "p_simple": float(s)} for p, r, s in zip(task1_prompts, task2_responses, task2_probabilities)],
        file, ensure_ascii=False, indent=2,
    )

del style_dpo_trainer, style_dpo_model, style_dpo_base
gc.collect()
torch.cuda.empty_cache()

## 6. Задача 3 — Reward Model

In [ ]:
from transformers import AutoModelForSequenceClassification
from trl import RewardConfig, RewardTrainer

reward_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": row["instruction"]}],
        "chosen": [{"role": "assistant", "content": row["chosen"]}],
        "rejected": [{"role": "assistant", "content": row["rejected"]}],
    }
    for row in good_bad_rows
])

def reward_lora_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS",
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=["score"],
    )

def load_qlora_reward_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_SOURCE,
        **model_revision_kwargs(),
        num_labels=1,
        quantization_config=QLORA_CONFIG,
        device_map={"": 0},
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    return prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}
    )

print("Reward preference pairs:", len(reward_dataset))

In [ ]:
set_seed(SEED)
reward_base_model = load_qlora_reward_model()
reward_args = RewardConfig(
    output_dir=str(RUNTIME_ROOT / "trainer_task3_reward_model"),
    seed=SEED,
    data_seed=SEED,
    max_length=MAX_LENGTH,
    num_train_epochs=1.0,
    max_steps=120,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataset_num_proc=2,
    center_rewards_coefficient=0.01,
)
reward_trainer = RewardTrainer(
    model=reward_base_model,
    args=reward_args,
    train_dataset=reward_dataset,
    processing_class=tokenizer,
    peft_config=reward_lora_config(),
)
reward_trainer.model.print_trainable_parameters()
reward_train_result = reward_trainer.train()
reward_trainer.save_model(str(RM_ADAPTER_DIR))
tokenizer.save_pretrained(RM_ADAPTER_DIR)
print("Reward Model train loss:", reward_train_result.training_loss)

In [ ]:
def sequence_reward_scores(
    model, tokenizer, prompts: list[str], completions: list[str], *, batch_size: int = 4
) -> np.ndarray:
    assert len(prompts) == len(completions) and prompts
    conversations = [
        tokenizer.apply_chat_template(
            [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": completion},
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
        for prompt, completion in zip(prompts, completions)
    ]
    device = next(model.parameters()).device
    scores = []
    was_training = model.training
    model.eval()
    try:
        for start in range(0, len(conversations), batch_size):
            batch_texts = conversations[start : start + batch_size]
            inputs = tokenizer(
                batch_texts,
                add_special_tokens=False,
                padding=True,
                truncation=False,
                return_tensors="pt",
            )
            assert int(inputs["attention_mask"].sum(dim=1).max()) <= MAX_LENGTH
            inputs = inputs.to(device)
            with torch.inference_mode():
                batch_scores = model(**inputs, use_cache=False).logits.squeeze(-1)
            scores.extend(batch_scores.float().cpu().tolist())
    finally:
        if was_training:
            model.train()
    return np.asarray(scores, dtype=np.float64)

task3_prompts = [row["prompt"] for row in public_quality_rows]
task3_chosen = [row["chosen"] for row in public_quality_rows]
task3_rejected = [row["rejected"] for row in public_quality_rows]
task3_chosen_rewards = sequence_reward_scores(
    reward_trainer.model, tokenizer, task3_prompts, task3_chosen
)
task3_rejected_rewards = sequence_reward_scores(
    reward_trainer.model, tokenizer, task3_prompts, task3_rejected
)
task3_accuracy, task3_correct, task3_total = strict_pairwise_accuracy(
    task3_chosen_rewards, task3_rejected_rewards
)
record_metric(
    "task_3_reward_model", task3_accuracy, "quality", correct=task3_correct, total=task3_total
)
print(f"Reward Model pairs: {task3_correct}/{task3_total}")

del reward_trainer, reward_base_model
gc.collect()
torch.cuda.empty_cache()

## 7. Задача 4 — DPO по качеству

In [ ]:
quality_preference_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": row["instruction"]}],
        "chosen": [{"role": "assistant", "content": row["chosen"]}],
        "rejected": [{"role": "assistant", "content": row["rejected"]}],
    }
    for row in good_bad_rows
])
quality_prompt_lengths = [
    len(tokenizer.apply_chat_template(record["prompt"], tokenize=True, add_generation_prompt=True))
    for record in quality_preference_dataset
]
quality_pair_lengths = [
    len(tokenizer.apply_chat_template(record["prompt"] + completion, tokenize=True))
    for record in quality_preference_dataset
    for completion in (record["chosen"], record["rejected"])
]
assert max(quality_prompt_lengths) <= 256, max(quality_prompt_lengths)
assert max(quality_pair_lengths) <= MAX_LENGTH, max(quality_pair_lengths)
print("Quality preference pairs:", len(quality_preference_dataset))
print("Quality token lengths (prompt/full):", max(quality_prompt_lengths), max(quality_pair_lengths))

In [ ]:
set_seed(SEED)
quality_dpo_base = load_qlora_causal_model()
quality_dpo_model = PeftModel.from_pretrained(
    quality_dpo_base, STYLE_DPO_ADAPTER_DIR, is_trainable=True, adapter_name="default"
)
quality_dpo_model.load_adapter(
    STYLE_DPO_ADAPTER_DIR, adapter_name="reference", is_trainable=False
)
quality_dpo_model.set_adapter("default")
assert set(quality_dpo_model.peft_config) == {"default", "reference"}
quality_dpo_model.print_trainable_parameters()

quality_dpo_args = DPOConfig(
    output_dir=str(RUNTIME_ROOT / "trainer_task4_quality_dpo"),
    seed=SEED,
    data_seed=SEED,
    model_adapter_name="default",
    ref_adapter_name="reference",
    beta=0.10,
    loss_type="sigmoid",
    max_length=MAX_LENGTH,
    max_prompt_length=256,
    num_train_epochs=1.0,
    max_steps=120,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataset_num_proc=2,
)
quality_dpo_trainer = DPOTrainer(
    model=quality_dpo_model,
    ref_model=None,
    args=quality_dpo_args,
    train_dataset=quality_preference_dataset,
    processing_class=tokenizer,
)
quality_dpo_result = quality_dpo_trainer.train()
quality_dpo_trainer.model.set_adapter("default")
quality_dpo_trainer.model.save_pretrained(
    QUALITY_DPO_ADAPTER_DIR, selected_adapters=["default"]
)
tokenizer.save_pretrained(QUALITY_DPO_ADAPTER_DIR)
print("Quality-DPO train loss:", quality_dpo_result.training_loss)

In [ ]:
task4_chosen_logprobs = mean_completion_logprobs(
    quality_dpo_trainer.model, tokenizer, task3_prompts, task3_chosen
)
task4_rejected_logprobs = mean_completion_logprobs(
    quality_dpo_trainer.model, tokenizer, task3_prompts, task3_rejected
)
task4_accuracy, task4_correct, task4_total = strict_pairwise_accuracy(
    task4_chosen_logprobs, task4_rejected_logprobs
)
record_metric(
    "task_4_dpo_quality", task4_accuracy, "quality", correct=task4_correct, total=task4_total
)
print(f"Quality-DPO pairs: {task4_correct}/{task4_total}")

del quality_dpo_trainer, quality_dpo_model, quality_dpo_base
gc.collect()
torch.cuda.empty_cache()

## 8. Задача 5 — SimPO

In [ ]:
from trl import CPOConfig, CPOTrainer

set_seed(SEED)
simpo_base = load_qlora_causal_model()
simpo_model = PeftModel.from_pretrained(
    simpo_base, STYLE_DPO_ADAPTER_DIR, is_trainable=True, adapter_name="default"
)
simpo_model.set_adapter("default")
assert set(simpo_model.peft_config) == {"default"}
simpo_model.print_trainable_parameters()

simpo_args = CPOConfig(
    output_dir=str(RUNTIME_ROOT / "trainer_task5_simpo"),
    seed=SEED,
    data_seed=SEED,
    loss_type="simpo",
    beta=2.0,
    simpo_gamma=1.0,
    cpo_alpha=0.0,
    max_length=MAX_LENGTH,
    max_prompt_length=256,
    num_train_epochs=1.0,
    max_steps=120,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataset_num_proc=2,
)
simpo_trainer = CPOTrainer(
    model=simpo_model,
    args=simpo_args,
    train_dataset=quality_preference_dataset,
    processing_class=tokenizer,
)
simpo_result = simpo_trainer.train()
simpo_trainer.model.save_pretrained(SIMPO_ADAPTER_DIR)
tokenizer.save_pretrained(SIMPO_ADAPTER_DIR)
print("SimPO train loss:", simpo_result.training_loss)

In [ ]:
task5_chosen_logprobs = mean_completion_logprobs(
    simpo_trainer.model, tokenizer, task3_prompts, task3_chosen
)
task5_rejected_logprobs = mean_completion_logprobs(
    simpo_trainer.model, tokenizer, task3_prompts, task3_rejected
)
task5_accuracy, task5_correct, task5_total = strict_pairwise_accuracy(
    task5_chosen_logprobs, task5_rejected_logprobs
)
record_metric(
    "task_5_simpo_quality", task5_accuracy, "quality", correct=task5_correct, total=task5_total
)
print(f"SimPO pairs: {task5_correct}/{task5_total}")

del simpo_trainer, simpo_model, simpo_base
gc.collect()
torch.cuda.empty_cache()

## 9. Итоговые метрики

Финальная таблица должна печатать точные значения и буквы интервалов для всех пяти задач.

In [ ]:
EXPECTED_TASKS = [
    "task_1_sft_style",
    "task_2_dpo_style",
    "task_3_reward_model",
    "task_4_dpo_quality",
    "task_5_simpo_quality",
]
assert list(METRICS) == EXPECTED_TASKS, (list(METRICS), EXPECTED_TASKS)
print("\nИтоговые метрики")
print(f"{'Задача':<28} {'Значение':>10}  Интервал")
for task in EXPECTED_TASKS:
    result = METRICS[task]
    print(f"{task:<28} {result['value']:>10.6f}  {result['interval']}")

with (RUNTIME_ROOT / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(METRICS, file, ensure_ascii=False, indent=2)
print("Буквы:", " ".join(METRICS[task]["interval"] for task in EXPECTED_TASKS))
print("Saved:", RUNTIME_ROOT / "metrics.json")